In [2]:
# requirements: pandas
import json, csv

INFILE = "climate_stations_full.json"
OUTFILE = "canadian_icao_stations.csv"



In [3]:
# fields to output
FIELDNAMES = ["station_name", "icao", "wmo_id", "latitude", "longitude", "elevation_m", "province", "reporting_type"]

def get_prop(feat, key, alt=None):
    props = feat.get("properties", {})
    return props.get(key, props.get(alt)) if props else None

with open(INFILE, "r", encoding="utf-8") as f:
    data = json.load(f)

# handle GeoJSON FeatureCollection or plain list
features = data.get("features") if isinstance(data, dict) else data




In [4]:
rows = []
for feat in features:
    props = feat.get("properties", {})
    # common ICAO fields: 'icao', 'ICAO', 'identifier'
    icao = (props.get("icao") or props.get("ICAO") or props.get("identifier") or "").strip()
    if not icao:
        continue
    # coordinates: GeoJSON geometry [lon,lat] or properties lat/lon
    geom = feat.get("geometry") or {}
    coords = geom.get("coordinates") if geom else None
    lon = lat = None
    if coords and isinstance(coords, (list,tuple)) and len(coords) >= 2:
        lon, lat = coords[0], coords[1]
    else:
        lat = props.get("latitude") or props.get("lat") or props.get("site_latitude")
        lon = props.get("longitude") or props.get("lon") or props.get("site_longitude")
    # other fields (try multiple possible property names)
    name = props.get("name") or props.get("station_name") or props.get("site_name") or ""
    wmo = props.get("wmo_identifier") or props.get("wmo_id") or props.get("wmo") or ""
    elev = props.get("elevation_m") or props.get("elevation") or props.get("altitude") or ""
    province = props.get("province") or props.get("prov") or props.get("state") or ""
    reporting_type = props.get("station_type") or props.get("reporting_type") or ""

    rows.append({
        "station_name": name,
        "icao": icao.upper(),
        "wmo_id": wmo,
        "latitude": lat,
        "longitude": lon,
        "elevation_m": elev,
        "province": province,
        "reporting_type": reporting_type
    })


In [5]:
# write CSV
with open(OUTFILE, "w", newline="", encoding="utf-8") as csvf:
    writer = csv.DictWriter(csvf, fieldnames=FIELDNAMES)
    writer.writeheader()
    for r in rows:
        writer.writerow(r)

print(f"Wrote {len(rows)} ICAO stations to {OUTFILE}")


Wrote 0 ICAO stations to canadian_icao_stations.csv
